# Limpieza del dataset

## 0. Importar librerías

In [32]:
import pandas as pd

## 1. Cargar datos

In [33]:
df = pd.read_csv("D:/BOOTCAMP/CARPETA_BARBARA/ML_MICROBIOME/ML/data/abundance_stoolsubset.csv")

df.shape

C:\Users\gonza\AppData\Local\Temp\ipykernel_25460\930949512.py:1: DtypeWarning: Columns (5,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("D:/BOOTCAMP/CARPETA_BARBARA/ML_MICROBIOME/ML/data/abundance_stoolsubset.csv")


(1989, 2339)

## 2. Limpieza del dataset

### 2.1. Seleccionar columnas de interés

In [34]:
# Seleccionar columnas de metadata interés
metadata_keep = [
    "dataset_name",
    "sampleID",
    "disease",
    "age",
    "gender",
    "country",
    "bmi"
]

In [35]:
# Conservar todas las columnas de especies
taxa_cols = [col for col in df.columns if col.startswith("k__")]

In [36]:
# Constriur el dataset "limpio"
df_clean = df[metadata_keep + taxa_cols].copy()

df_clean.shape

(1989, 2135)

### 2.2. Formatear el dataset

#### 2.2.1. Unificar missing values

Identificar patrones `"nd"`, `"ND"`, `"-"`, `" -"`, `""`, `" "`, `"na"`, `"NA"`, `"nan"`, `"None"` y reemplazar por `NA` para un mejor análisis.

In [37]:
missing_values = ["nd", "ND", "-", " -", "", " ", "na", "NA", "nan", "None"]

df_clean = df_clean.replace(missing_values, pd.NA)

#### 2.2.2. Convertir columnas numéricas

Convertir a numeric las columnas `"age"` y `"bmi"`

In [38]:
df_clean["age"] = pd.to_numeric(df_clean["age"], errors="coerce")
df_clean["bmi"] = pd.to_numeric(df_clean["bmi"], errors="coerce")

#### 2.2.3. Normalizar columnas de texto

In [39]:
# 1. Convertir la columna a tipo texto: .astype("string")
# 2. Quitar espacios al principio y final: .str.strip()
# 3. Convertir el texto a minúsculas: .str.lower()

# 2.2.3. Normalizar columnas de texto
text_cols = ["dataset_name", "sampleID", "disease", "gender", "country"]

for col in text_cols:
    df_clean[col] = df_clean[col].astype("string").str.strip().str.lower()

#### 2.2.4. Unificar etiquetas de datos clínicos de la variable `disease`

In [40]:
df_clean["disease"] = df_clean["disease"].replace({
    "leaness": "leanness",
    "underweight": "leanness",
    "obese": "obesity"
})

In [41]:
df_clean["disease"].value_counts()

disease
n                             944
t2d                           223
obesity                       169
ibd_ulcerative_colitis        148
cirrhosis                     118
leanness                       90
stec2-positive                 52
impaired_glucose_tolerance     49
cancer                         48
n_relative                     47
small_adenoma                  26
ibd_crohn_disease              25
large_adenoma                  13
overweight                     10
Name: count, dtype: Int64

#### 2.2.5. Gestionar nulos en metadata categórica

In [42]:
df_clean["country"] = df_clean["country"].fillna("unknown")
df_clean["gender"] = df_clean["gender"].fillna("unknown")

#### 2.2.6. Convertir taxones a numérico

In [43]:
df_clean[taxa_cols] = df_clean[taxa_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

df_clean[taxa_cols] = df_clean[taxa_cols].fillna(0)

#### 2.2.7. Revisar la calidad de metadata

In [44]:
df_clean[metadata_keep].isna().sum().sort_values(ascending=False)

age             549
bmi             367
disease          27
sampleID          0
dataset_name      0
gender            0
country           0
dtype: int64

In [45]:
df_clean["disease"].value_counts(dropna=False)

disease
n                             944
t2d                           223
obesity                       169
ibd_ulcerative_colitis        148
cirrhosis                     118
leanness                       90
stec2-positive                 52
impaired_glucose_tolerance     49
cancer                         48
n_relative                     47
<NA>                           27
small_adenoma                  26
ibd_crohn_disease              25
large_adenoma                  13
overweight                     10
Name: count, dtype: Int64

In [46]:
df_clean["sampleID"].duplicated().sum()

np.int64(243)

#### 2.2.8. Depurar duplicados por `sampleID`

In [47]:
# Crear una copia de df_clean para trabajar
df_work = df_clean.copy()

In [48]:
# Eliminar filas completamente idénticas, si existieran
df_work = df_work.drop_duplicates().copy()

In [49]:
# Eliminar registros sin identificador o sin disease
df_work = df_work[
    df_work["sampleID"].notna() &
    df_work["disease"].notna()
].copy()

In [50]:
# Detectar sampleID con disease contradictorio
disease_counts_by_sample = (
    df_work
    .groupby("sampleID")["disease"]
    .nunique(dropna=False)
)

conflict_sample_ids = disease_counts_by_sample[
    disease_counts_by_sample > 1
].index

len(conflict_sample_ids)

142

In [51]:
# Eliminar sampleID conflictivos
df_work = df_work[
    ~df_work["sampleID"].isin(conflict_sample_ids)
].copy()

df_work.shape

(1616, 2135)

In [52]:
# Para sampleID repetidos sin conflicto, conservar la fila con metadata más completa
metadata_for_completeness = [
    "dataset_name",
    "disease",
    "age",
    "gender",
    "country",
    "bmi"
]

df_work["metadata_complete_count"] = (
    df_work[metadata_for_completeness]
    .notna()
    .sum(axis=1)
)

df_work = df_work.sort_values(
    by=["sampleID", "metadata_complete_count"],
    ascending=[True, False]
)

df_clean_dedup = df_work.drop_duplicates(
    subset="sampleID",
    keep="first"
).copy()

df_clean_dedup = df_clean_dedup.drop(columns="metadata_complete_count")

df_clean_dedup.shape

(1577, 2135)

#### 2.2.9. Comprobaciones finales

In [59]:
print("Dimensiones del dataset limpio:", df_clean_dedup.shape)
print("--------------------------")

print("SampleID duplicados:")
print(df_clean_dedup["sampleID"].duplicated().sum())
print("--------------------------")

print("Valores nulos en metadata:")
print(df_clean_dedup[metadata_keep].isna().sum().sort_values(ascending=False))
print("--------------------------")

print("Distribución de la variable disease:")
print(df_clean_dedup["disease"].value_counts(dropna=False))

Dimensiones del dataset limpio: (1577, 2135)
--------------------------
SampleID duplicados:
0
--------------------------
Valores nulos en metadata:
age             356
bmi             331
dataset_name      0
disease           0
sampleID          0
gender            0
country           0
dtype: int64
--------------------------
Distribución de la variable disease:
disease
n                             710
t2d                           223
ibd_ulcerative_colitis        148
cirrhosis                     118
obesity                        75
impaired_glucose_tolerance     49
cancer                         48
n_relative                     47
stec2-positive                 43
leanness                       42
small_adenoma                  26
ibd_crohn_disease              25
large_adenoma                  13
overweight                     10
Name: count, dtype: Int64


## 3. Dataset limpio

In [62]:
processed = "D:/BOOTCAMP/CARPETA_BARBARA/ML_MICROBIOME/ML/data/processed/abundance_stoolsubset_clean.csv"

df_clean_dedup.to_csv(processed, index=False)